# 12 · Entanglement swapping — repeater chains

Direct entanglement distribution dies exponentially with distance. A **quantum
repeater** splits the path into short elementary links and stitches them together with
**entanglement swapping**: a middle station holding one half of each adjacent link
performs a Bell-state measurement (BSM) on its two qubits. That projects the two *outer*
qubits — which never interacted — into a Bell state identified by the station's 2-bit
**herald** (m1, m2); applying the Pauli correction X^m2·Z^m1 at one end restores Φ+.

The register's `fidelity` knob f is the Werner parameter: each elementary pair is
ρ = f·|Φ+⟩⟨Φ+| + (1−f)·I/4. Swapping composes Werner states multiplicatively, so an
L-link chain obeys

    w_chain = f^L   →   QBER = (1 − f^L)/2,   F = (1 + 3·f^L)/4,   CHSH S = 2√2·f^L

Nothing in the code hard-codes that law — it has to **emerge** from the BSM circuit and
the heralded corrections. This notebook checks exactly that:

1. end-to-end QBER vs chain length against the law,
2. the CHSH Bell violation degrading with hops,
3. the herald channel is *load-bearing* (skip the correction → useless pair),
4. the same chain distributed across **three OS processes**, heralds crossing a real
   (loopback) link — the FABRIC variant (`12_repeater_fabric`) then puts those links on
   the WAN with the station on the switch node.

Run this on the SeQUeNCe-env kernel (`sequence`, `numpy`, `matplotlib`).

## 1 · Setup — in-process chain smoke test

In [ ]:
import sys, os, json, subprocess, time, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PKG = PROJECT_DIR / 'qne-sequence'
for p in (str(PROJECT_DIR), str(PKG)):
    if p not in sys.path:
        sys.path.insert(0, p)

from qne_sequence.repeater import run_chain_session, chain_qber, chain_chsh

r = run_chain_session(num_nodes=3, num_pairs=2000, fidelity=0.95, seed=7)
print(f"3-node chain: delivered={r.delivered}  swaps={r.swaps}")
print(f"  QBER {r.qber:.4f}  (law {r.qber_pred:.4f})")
print(f"  F̂   {r.fidelity_est:.4f}  (law {r.fidelity_pred:.4f})")
print(f"  heralds {r.heralds}")

## 2 · QBER vs chain length — the Werner-chain law

Each point is a full session (create L pairs → swap at every intermediate node →
heralded correction → matching-basis measurement at the ends). The curves are the law
QBER = (1 − f^L)/2 — the physics test is that the circuit lands on them.

In [ ]:
fids = (0.90, 0.95, 0.98)
nodes = range(2, 7)                      # L = 1..5 links
fig, ax = plt.subplots(figsize=(7, 4.5))
for f in fids:
    measured = [run_chain_session(n, 4000, fidelity=f, seed=11 + n).qber
                for n in nodes]
    L = np.array([n - 1 for n in nodes])
    ax.plot(L, [chain_qber(f, int(l)) for l in L], '--', alpha=0.6,
            label=f'law  f={f}')
    ax.plot(L, measured, 'o', ms=6, label=f'measured  f={f}')
ax.set_xlabel('links L (nodes − 1)'); ax.set_ylabel('end-to-end QBER')
ax.set_title('Swapped-chain QBER follows (1 − f^L)/2')
ax.set_xticks(list(L)); ax.legend(ncol=2, fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3 · CHSH across the chain — where the Bell violation dies

S = 2√2·f^L: every extra noisy hop multiplies the violation down. Past S ≤ 2 the end-to-end
pair no longer certifies entanglement — the operational limit for an E91-style key over
that chain.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for f in (0.90, 0.97):
    s_meas = [run_chain_session(n, 6000, fidelity=f, mode='chsh', seed=23 + n).chsh_s
              for n in nodes]
    L = np.array([n - 1 for n in nodes])
    ax.plot(L, [chain_chsh(f, int(l)) for l in L], '--', alpha=0.6, label=f'law  f={f}')
    ax.plot(L, s_meas, 's', ms=6, label=f'measured  f={f}')
ax.axhline(2.0, color='r', lw=1, label='classical bound S = 2')
ax.axhline(2 * math.sqrt(2), color='gray', lw=1, ls=':', label='Tsirelson 2√2')
ax.set_xlabel('links L'); ax.set_ylabel('CHSH S')
ax.set_title('Bell violation across a swapped chain: S = 2√2·f^L')
ax.set_xticks(list(L)); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4 · Control — the herald channel is load-bearing

Skip the Pauli correction and the end pair, averaged over the four (uncommunicated)
heralds, is an even Bell mixture: QBER → 1/2, no correlations at all. Entanglement
swapping *without the classical herald* distributes nothing — this is why the herald
link's latency matters on a real network.

In [ ]:
ok = run_chain_session(3, 3000, fidelity=1.0, seed=41)
no = run_chain_session(3, 3000, fidelity=1.0, apply_correction=False, seed=41)
print(f"perfect links, corrected:   QBER = {ok.qber:.4f}   (phi+ restored)")
print(f"perfect links, NO herald:   QBER = {no.qber:.4f}   (even Bell mixture)")

## 5 · Distributed — three processes, real herald traffic

Same chain, now across three OS processes and three TCP links:

| link | carries |
|------|---------|
| alice ↔ station | swap plan + the BSM ops against the shared register |
| station → bob | **the heralds** (the multi-hop classical hop) |
| alice ↔ bob | basis announcement, QBER sample, CHSH bits, Cascade parities |

Bob applies X^m2·Z^m1 from the heralds *before* measuring, then the standard QKD tail
runs (sift → sample QBER → Cascade → Toeplitz PA) and both ends extract the identical
secret.

In [ ]:
_PORT = [57740]

def run_chain_distributed(num_pairs=4000, fidelity=0.95, chain_mode='bbm92',
                          extra=(), seed_base=1):
    """Launch bob + repeater + alice on loopback; return {role: result}."""
    _PORT[0] += 4
    port = _PORT[0]
    env = dict(os.environ)
    env['PYTHONPATH'] = f"{PKG}{os.pathsep}{PROJECT_DIR}"
    def spawn(role, seed):
        return subprocess.Popen(
            [sys.executable, '-m', 'qne_sequence.node_runner',
             '--role', role, '--name', role, '--protocol', 'repeater',
             '--chain-mode', chain_mode, '--host', '127.0.0.1',
             '--port', str(port), '--num-pairs', str(num_pairs),
             '--fidelity', str(fidelity), '--sample-fraction', '0.2',
             '--seed', str(seed), *extra],
            cwd=str(PKG), env=env,
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    procs = {'bob': spawn('bob', seed_base + 1),
             'repeater': spawn('repeater', seed_base + 2),
             'alice': spawn('alice', seed_base)}
    out = {}
    for who, p in procs.items():
        o, e = p.communicate(timeout=120)
        line = next((ln for ln in o.splitlines() if ln.startswith('{')), '')
        assert line, f"{who} produced no result:\n{e}"
        out[who] = json.loads(line)
    return out

res = run_chain_distributed(num_pairs=4000, fidelity=0.95)
a, b, st = res['alice'], res['bob'], res['repeater']
print(f"alice : QBER {a['qber']:.4f} (law {a['qber_pred']:.4f})  "
      f"sifted {a['sifted_bits']}  reconciled {a['reconciled']}  "
      f"secure_key_bits {a['secure_key_bits']}")
print(f"bob   : key matches alice bit-for-bit: {a['key'] == b['key']}")
print(f"station: swaps {st['swaps']}  heralds {st['heralds']}  "
      f"frames tx/rx {st['tx_frames']}/{st['rx_frames']}")

### CHSH across processes — a Bell violation certified over the network

In [ ]:
res = run_chain_distributed(num_pairs=6000, fidelity=0.95, chain_mode='e91')
a = res['alice']
print(f"CHSH S = {a['chsh_s']:.3f}  (law {a['chsh_pred']:.3f}, "
      f"classical bound 2.0) over {a['chsh_pairs']} pairs")
print(f"violation: {'YES' if a['chsh_s'] > 2 else 'no'}  |  "
      f"keys match: {a['key'] == res['bob']['key']}")

## Notes → FABRIC

- `12_repeater_fabric` runs exactly this across the slice: the station lives on the
  **switch node** (BMv2 swapped for a Linux bridge + station IP via
  `deploy_fabric.setup_repeater_bridge`), so the heralds traverse a real WAN segment.
- The herald link is the latency lever: `apply_classical_netem` on the switch's
  bob-side interface delays exactly the herald traffic.
- Finite-key over a chain is brutal: at f = 0.95 (QBER ≈ 4.9%) a ~6k-bit block
  finite-keys to **zero** — noisy chains need long blocks (see notebook 13).